In [1]:
import torch
import torch.nn.functional as F 
from torch import nn
from torch import optim
from torch.utils.data import DataLoader

from torchvision import transforms
from torchvision import models
from torchvision.datasets import ImageFolder

from PIL import Image

import numpy as np

In [2]:
if torch.cuda.is_available():
    dev = "cuda:0"
elif torch.backends.mps.is_available():
    dev = "mps"
else:
    dev = "cpu"
device = torch.device(dev)
device

device(type='mps')

In [3]:
data_transforms = transforms.Compose([
    transforms.Resize((224,224)),   # resize the input to 224x224
    transforms.ToTensor(),          # put the input to tensor format
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # normalize the input based on images from ImageNet
])

In [4]:
train_ds = ImageFolder(root="intel-image-classification/seg_train", transform=data_transforms)
mini_batch_size = 512
train_dl = DataLoader(train_ds, batch_size=mini_batch_size, shuffle=True)

In [5]:
alexnet = models.alexnet(pretrained=True)

/Users/wick/ML-notebooks/.venv/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/wick/ML-notebooks/.venv/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [6]:
class WrappedDataLoader:
    def __init__(self, dl, func):
        self.dl = dl
        self.func = func

    def __len__(self):
        return len(self.dl)

    def __iter__(self):
        for b in self.dl:
            yield (self.func(*b))

In [7]:
def put_to_gpu(x, y):
    return x.to(device), y.to(device)

In [8]:
def fit(epochs, model, optimizer, train_dl):
    loss_func = nn.CrossEntropyLoss()

    # loop over epochs
    for epoch in range(epochs):
        model.train()

        # loop over mini-batches
        for X_mb, y_mb in train_dl:
            y_hat = model(X_mb)

            loss = loss_func(y_hat, y_mb)
            loss.backward()

            optimizer.step()
            optimizer.zero_grad()

        print('epoch {}, loss {}'.format(epoch, loss.item()))

    print('Finished training')

    return model

In [ ]:
def finetune(model, train_dl):
    for param in model.parameters():
        param.requires_grad = False

    new_layers = nn.Sequential(
        nn.Dropout(p=0.5, inplace=False),
        nn.Linear(in_features=9216, out_features=4096, bias=True),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.5, inplace=False),
        nn.Linear(in_features=4096, out_features=1000, bias=True),
        nn.ReLU(inplace=True),
        nn.Linear(in_features=1000, out_features=6, bias=True),
    )
    model.classifier = new_layers

    optimizer = optim.Adam(model.parameters())

    train_dl = WrappedDataLoader(train_dl, put_to_gpu)
    model = model.to(device)

    epochs = 10
    trained_model = fit(epochs, model, optimizer, train_dl)

    return trained_model

In [10]:
trained_model = finetune(alexnet, train_dl)

epoch 0, loss 0.2734420597553253
epoch 1, loss 0.28411081433296204
epoch 2, loss 0.18461085855960846
epoch 3, loss 0.2472962886095047
epoch 4, loss 0.09215600043535233
epoch 5, loss 0.10368485003709793
epoch 6, loss 0.11029426753520966
epoch 7, loss 0.06329258531332016
epoch 8, loss 0.12433964014053345
epoch 9, loss 0.15696559846401215
Finished training


In [11]:
trained_model.eval()

AlexNet(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (avgpool): AdaptiveAvgPool2d(output_size=(6, 6))
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=9216, out_features=4096, bias=True)
 

In [12]:
test_image = Image.open("intel-image-classification/seg_test/sea/21191.jpg")
print("original image's shape: " + str(test_image.size))
transformed_img = data_transforms(test_image)
print("transformed image's shape: " + str(transformed_img.shape))
# form a batch with only one image
batch_img = torch.unsqueeze(transformed_img, 0)
print("image batch's shape: " + str(batch_img.shape))

output = trained_model.to('cpu')(batch_img)

print("output vector's shape: " + str(output.shape))
percentage = F.softmax(output, dim=1)[0] * 100.0
_, indices = torch.sort(output, descending=True)

# map the class no. to the corresponding label
with open('intel-image-classification/class_names_Intel.txt') as labels:
    classes = [i.strip() for i in labels.readlines()]
results = [(classes[i], percentage[i].item()) for i in indices[0][:5]]

for i in range(3):
    print('{}: {:.4f}%'.format(results[i][0], results[i][1]))

original image's shape: (150, 150)
transformed image's shape: torch.Size([3, 224, 224])
image batch's shape: torch.Size([1, 3, 224, 224])
output vector's shape: torch.Size([1, 6])
sea: 99.6888%
glacier: 0.3112%
mountain: 0.0000%


In [13]:
def evaluate(model, test_loader):    
    model.eval()
    accuracy = 0
    with torch.no_grad():
        for X, y in test_loader:
            y_hat = model(X)
            y_hat = F.softmax(y_hat, dim=1).cpu().numpy()
            y_hat = np.argmax(y_hat, axis=1)
            accuracy += (y_hat == y.cpu().numpy()).mean()
    accuracy /= len(test_loader)

    return accuracy

In [14]:
test_ds = ImageFolder(root="intel-image-classification/seg_test", transform=data_transforms)
test_dl = DataLoader(test_ds, batch_size=mini_batch_size*2)
accuracy = evaluate(trained_model.to(device), WrappedDataLoader(test_dl, put_to_gpu))
print("accuracy: " + str(accuracy))

accuracy: 0.9186608237044819
